# short_volshock_recipe_scope_audit_v1

Audit the reward/checkpoint recipe contract before any new PPO or actor retention step. This notebook consumes the `short_volshock_teacher_reward_checkpoint_screen_v1` contract, checks final walk-forward window coverage, checks whether selected daily paths are safe for training scope, and writes a machine-readable readiness decision.

## Experiment Contract

- Baselines: canonical SOXLBacktest qlib reference `soxl_backtest_policy_reimplementation_v1`; direct comparison `short_volshock_teacher_reward_checkpoint_screen_v1`.
- Role: `diagnose` / pre-training gate audit.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `reward`, `checkpoint_selection`, `model_selection`, `target_scope`.
- Hypothesis: the near-miss short-volshock teacher may be usable only after fold-local scope and missing-window coverage gates pass.
- Acceptance: training remains blocked unless all final windows have official replay coverage and zero-overlap fold-local target manifests.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "soxl_quant").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "soxl_quant").exists():
    raise RuntimeError("could not locate repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.reward_checkpoint_screen import audit_reward_checkpoint_recipe_scope

EXPERIMENT = "short_volshock_recipe_scope_audit_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
REWARD_CARD = ROOT / "runtime" / "reward" / "card" / "short_volshock_teacher_reward_checkpoint_screen_v1"
TEACHER_CARD = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1"
SCREEN_CARD = ROOT / "runtime" / "model_selection" / "card" / "official_replay_metric_aligned_checkpoint_bc_screen_v1"
CARD.mkdir(parents=True, exist_ok=True)
created_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
created_at


'2026-06-16T04:43:59+00:00'

## Load Source Artifacts

The selected daily paths are official replay diagnostics. They are not fold-local training labels unless a separate zero-overlap target manifest exists.

In [2]:
sources = {
    "contract": REWARD_CARD / "reward_checkpoint_recipe_contract.json",
    "reward_summary": REWARD_CARD / "summary.json",
    "teacher_window_coverage": TEACHER_CARD / "teacher_window_coverage.csv",
    "selected_teacher_fold_metrics": TEACHER_CARD / "selected_teacher_fold_metrics.csv",
    "teacher_selection": TEACHER_CARD / "selection.json",
    "screen_selection": SCREEN_CARD / "selection.json",
}
source_manifest = pd.DataFrame(
    [
        {
            "name": name,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "bytes": path.stat().st_size if path.exists() else 0,
        }
        for name, path in sources.items()
    ]
)
missing_sources = source_manifest.loc[~source_manifest["exists"], "name"].tolist()
if missing_sources:
    raise FileNotFoundError(f"missing source artifacts: {missing_sources}")

contract = json.loads(sources["contract"].read_text(encoding="utf-8"))
reward_summary = json.loads(sources["reward_summary"].read_text(encoding="utf-8"))
teacher_selection = json.loads(sources["teacher_selection"].read_text(encoding="utf-8"))
screen_selection = json.loads(sources["screen_selection"].read_text(encoding="utf-8"))
teacher_window_coverage = pd.read_csv(sources["teacher_window_coverage"])
selected_teacher_fold_metrics = pd.read_csv(sources["selected_teacher_fold_metrics"])
source_manifest.to_csv(CARD / "source_manifest.csv", index=False)
source_manifest


,name,path,exists,bytes
0,contract,runtime/reward/card/short_volshock_teacher_rew...,True,2556
1,reward_summary,runtime/reward/card/short_volshock_teacher_rew...,True,1968
2,teacher_window_coverage,runtime/model_selection/card/short_volshock_me...,True,211
3,selected_teacher_fold_metrics,runtime/model_selection/card/short_volshock_me...,True,2299
4,teacher_selection,runtime/model_selection/card/short_volshock_me...,True,3830
5,screen_selection,runtime/model_selection/card/official_replay_m...,True,3645


## Scope Audit

No fold-local target manifest is provided yet, so the audit should keep training blocked even though the contract has now been consumed by code.

In [3]:
audit = audit_reward_checkpoint_recipe_scope(
    contract,
    teacher_window_coverage=teacher_window_coverage,
    selected_fold_metrics=selected_teacher_fold_metrics,
    fold_local_target_manifest=None,
)
coverage_rows = audit["coverage_rows"]
scope_rows = audit["scope_rows"]
required_gate_rows = audit["required_gate_rows"]
audit_summary = audit["summary"]

coverage_rows.to_csv(CARD / "coverage_audit.csv", index=False)
scope_rows.to_csv(CARD / "scope_audit.csv", index=False)
required_gate_rows.to_csv(CARD / "required_gate_rows.csv", index=False)

audit_summary


{'decision': 'recipe_scope_audit_blocks_training',
 'selected_candidate': 'teacher_short_volshock_cap05_hit2',
 'training_allowed': False,
 'coverage_gate': False,
 'scope_gate': False,
 'missing_final_required_windows': ['wf_2020'],
 'required_window_count': 3,
 'target_ready_window_count': 0}

## Persist Decision

This notebook may only unblock training after coverage and scope gates pass. It does not create an actor, does not promote a near-miss, and does not authorize PPO.

In [4]:
training_allowed = bool(audit_summary["training_allowed"])
decision = str(audit_summary["decision"])
selection = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "role": "diagnose",
    "canonical_baseline": contract.get("reference_profile") or contract.get("target_contract", {}).get("reference_profile"),
    "direct_comparison_baseline": "short_volshock_teacher_reward_checkpoint_screen_v1",
    "source_contract": str(sources["contract"].relative_to(ROOT)),
    "selected_candidate": contract.get("selected_candidate"),
    "changed_layer": ["reward", "checkpoint_selection", "model_selection", "target_scope"],
    "target_windows": contract.get("final_required_windows", []),
    "accepted_tradeoff": "none before all final windows have official replay coverage and zero-overlap fold-local targets",
    "hypothesis": "short-volshock teacher can only feed the next training loop after scope and missing-window gates pass",
    "acceptance_criteria": {
        "coverage_gate": True,
        "scope_gate": True,
        "training_allowed": True,
        "eval_overlap_rows_required": contract.get("leakage_guard", {}).get("eval_overlap_rows_required", 0),
        "strict_final_retention_rerun_required": True,
    },
    "closest_historical_success_case": "short_volshock_cap05_hit2 passed wf_2022 official replay and was near-miss on wf_2023_2024",
    "closest_historical_failure_case": "official replay metric-aligned screen found no candidate with full final-window coverage or final retained actor status",
    "why_not_duplicate": "this consumes the reward checkpoint contract and audits target scope/readiness, rather than rescoring official replay candidates",
    "decision": decision,
    "training_allowed": training_allowed,
    "ppo_allowed_next": training_allowed,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_mechanism": "generate_fold_local_short_volshock_targets_with_wf2020_coverage" if not training_allowed else "run_strict_final_walkforward_retention_gate_after_training",
    "hard_blockers": required_gate_rows.loc[required_gate_rows["status"].eq("blocked"), "gate"].tolist(),
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": decision,
    "selected_candidate": contract.get("selected_candidate"),
    "training_allowed": training_allowed,
    "ppo_allowed_next": training_allowed,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "coverage_gate": bool(audit_summary["coverage_gate"]),
    "scope_gate": bool(audit_summary["scope_gate"]),
    "missing_final_required_windows": audit_summary["missing_final_required_windows"],
    "required_window_count": int(audit_summary["required_window_count"]),
    "target_ready_window_count": int(audit_summary["target_ready_window_count"]),
    "blocked_gates": selection["hard_blockers"],
    "reference_profile": selection["canonical_baseline"],
    "source_teacher_decision": contract.get("source_teacher_decision"),
    "source_teacher_selected_candidate": reward_summary.get("selected_candidate"),
    "screen_selected_missing_final_required_windows": screen_selection.get("selected_missing_final_required_windows"),
    "next_mechanism": selection["next_mechanism"],
    "sidecars": [
        "source_manifest.csv",
        "coverage_audit.csv",
        "scope_audit.csv",
        "required_gate_rows.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}
conclusion_lines = [
    f"# {EXPERIMENT} Conclusion",
    "",
    f"Decision: `{decision}`.",
    "",
    f"Coverage gate: `{summary['coverage_gate']}`; scope gate: `{summary['scope_gate']}`; training_allowed: `{training_allowed}`.",
    f"Missing final windows: {', '.join(summary['missing_final_required_windows']) or 'none'}.",
    f"Blocked gates: {', '.join(summary['blocked_gates']) or 'none'}.",
    "",
    "The short-volshock teacher remains reference-only. Selected official replay daily paths must not be copied into training labels.",
    "",
    f"Next action: `{summary['next_mechanism']}`.",
]
conclusion_md = "\n".join(conclusion_lines) + "\n"

(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "conclusion.md").write_text(conclusion_md, encoding="utf-8")
summary


{'experiment': 'short_volshock_recipe_scope_audit_v1',
 'created_at': '2026-06-16T04:43:59+00:00',
 'decision': 'recipe_scope_audit_blocks_training',
 'selected_candidate': 'teacher_short_volshock_cap05_hit2',
 'training_allowed': False,
 'ppo_allowed_next': False,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'coverage_gate': False,
 'scope_gate': False,
 'missing_final_required_windows': ['wf_2020'],
 'required_window_count': 3,
 'target_ready_window_count': 0,
 'blocked_gates': ['fold_local_target_scope_audit',
  'add_wf_2020_official_replay_coverage',
  'rerun_strict_final_walkforward_retention_gate'],
 'reference_profile': 'soxl_backtest_policy_reimplementation_v1',
 'source_teacher_decision': 'short_volshock_metric_aligned_teacher_ready_no_promotion',
 'source_teacher_selected_candidate': 'teacher_short_volshock_cap05_hit2',
 'screen_selected_missing_final_required_windows': 'wf_2020',
 'next_mechanism': 'generate_fold_local_short_volshock_targets_with_wf2020_c

In [5]:
from IPython.display import Markdown

Markdown(conclusion_md)


# short_volshock_recipe_scope_audit_v1 Conclusion

Decision: `recipe_scope_audit_blocks_training`.

Coverage gate: `False`; scope gate: `False`; training_allowed: `False`.
Missing final windows: wf_2020.
Blocked gates: fold_local_target_scope_audit, add_wf_2020_official_replay_coverage, rerun_strict_final_walkforward_retention_gate.

The short-volshock teacher remains reference-only. Selected official replay daily paths must not be copied into training labels.

Next action: `generate_fold_local_short_volshock_targets_with_wf2020_coverage`.
